# Lab 04: Spark Streaming

**Môn học:** Nhập môn Dữ liệu lớn  - Nhóm 4          
**Repository lựa chọn:** `huggingface/peft`

## 0. Chuẩn bị notebook

Các file evidence được đặt trong thư mục:

```text
evidence/
output/
configs/
jobs/
```

Cell dưới đây dò tìm project root có chứa `evidence/`

In [ ]:
from pathlib import Path
import json
from pprint import pprint


def find_project_root(start: Path = Path.cwd()) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "evidence").exists():
            return candidate
    return current


PROJECT_ROOT = find_project_root()
EVIDENCE_DIR = PROJECT_ROOT / "evidence"
OUTPUT_DIR = PROJECT_ROOT / "output"
CONFIG_DIR = PROJECT_ROOT / "configs"
JOBS_DIR = PROJECT_ROOT / "jobs"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("EVIDENCE_DIR =", EVIDENCE_DIR)

PROJECT_ROOT = /Users/hvpu/Downloads/group4
EVIDENCE_DIR = /Users/hvpu/Downloads/group4/evidence


In [ ]:
def read_text(path, max_chars=None):
    path = Path(path)
    if not path.exists():
        return f"[MISSING] {path}"
    text = path.read_text(encoding="utf-8", errors="replace")
    if max_chars is not None and len(text) > max_chars:
        return text[:max_chars] + f"\n... [truncated, total {len(text)} chars]"
    return text


def read_json(path):
    path = Path(path)
    if not path.exists():
        print(f"[MISSING] {path}")
        return None
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def show_file(path, max_chars=4000):
    print(read_text(path, max_chars=max_chars))

## 1. Repository Cloning and File Discovery

Shallow clone repository:

```bash
git clone --depth 1 https://github.com/huggingface/peft.git peft
```

Commit sử dụng: `79f4c362248d3b3b4bc2ed24704ed3183528c53f`

Đếm số lượng file `.py` theo hai chế độ (đã loại trừ các file sinh tự động):

- Không tính thư mục test
- Có tính thư mục test

In [4]:
discover_no_tests = read_json(EVIDENCE_DIR / "discover_no_tests.json")
discover_include_tests = read_json(EVIDENCE_DIR / "discover_include_tests.json")

print("Discovery không tính tests:")
pprint(discover_no_tests)

print("\nDiscovery có tính tests:")
pprint(discover_include_tests)

Discovery không tính tests:
{'by_top_level_dir': {'docs': 1,
                      'examples': 93,
                      'method_comparison': 17,
                      'scripts': 8,
                      'setup.py': 1,
                      'src': 245},
 'repo_root': '/Users/hvpu/Downloads/group4/peft',
 'sample_files': ['docs/source/_config.py',
                  'examples/KappaTune/experiments_kappatune_peft.py',
                  'examples/adamss_finetuning/glue_adamss_asa_example.py',
                  'examples/adamss_finetuning/glue_adamss_asa_manual_example.py',
                  'examples/adamss_finetuning/image_classification_adamss_asa.py',
                  'examples/alora_finetuning/alora_finetuning.py',
                  'examples/arrow_multitask/arrow_phi3_mini.py',
                  'examples/bdlora_finetuning/chat.py',
                  'examples/beft_finetuning/beft_finetuning.py',
                  'examples/boft_controlnet/__init__.py'],
 'total_files': 365}

Discove

### Kết quả discovery:
- Không tính tests: 365 file Python
- Có tính tests: 431 file Python

Nhóm sử dụng tập file Python không tính test files cho các bước parse và sinh event.


## 2. Incremental CPG Parser Service

Parser Service sinh ra các loại event chính:

| Loại event | Ý nghĩa |
|---|---|
| `node` | Biểu diễn node trong AST/CPG |
| `edge` | Biểu diễn quan hệ giữa các node |
| `metadata` | Thông tin tổng hợp của từng file |
| `error` | Lỗi parser nếu có |

Các thành phần CPG được trích xuất gồm:

- AST nodes: node cú pháp của chương trình.
- AST edges: quan hệ cha-con trong cây cú pháp.
- CFG edges: luồng điều khiển.
- DFG edges: luồng dữ liệu.
- Call edges: quan hệ gọi hàm.

In [ ]:
parse_summary = read_json(EVIDENCE_DIR / "parse_summary.json")
pprint(parse_summary)

{'edge_type_counts': {'AST_CHILD': 246361,
                      'CALLS': 23036,
                      'CFG_ELSE': 3,
                      'CFG_EXCEPT': 61,
                      'CFG_FALSE': 783,
                      'CFG_FINALLY': 6,
                      'CFG_LOOP': 2630,
                      'CFG_NEXT': 20179,
                      'CFG_TRUE': 4169,
                      'CFG_TRY': 138,
                      'DFG': 55961},
 'event_file': 'output/peft-events.jsonl',
 'event_lines': 604529,
 'sample_topics_written_to': 'evidence/sample_events_by_topic.json',
 'top_node_kinds': {'Assign': 15346,
                    'Attribute': 17148,
                    'BinOp': 2780,
                    'Call': 23018,
                    'Compare': 4118,
                    'Constant': 49591,
                    'Dict': 1540,
                    'Expr': 6692,
                    'FormattedValue': 2289,
                    'FunctionDef': 2528,
                    'If': 5841,
                    'I

### Kết quả thực thi:

Parser xử lý các files `.py` thu được các kết quả sau:
- 365 files
- 250,837 node events
- 353,327 edge events
- 365 metadata events
- 0 errors

Số parser errors là 0, cho thấy parser có thể xử lý ổn định trên source code chính của repository


## 3. Kafka Topic Design

Task 3 định nghĩa lớp trung gian Kafka cho toàn bộ pipeline. Thay vì đẩy mọi dữ liệu vào một topic chung, pipeline tách event theo loại dữ liệu để mỗi downstream sink chỉ consume đúng phần nó cần.

| Kafka topic | Vai trò trong pipeline | Downstream chính |
|---|---|---|
| `peft.cpg.nodes` | Chứa node events của Code Property Graph | Neo4j |
| `peft.cpg.edges` | Chứa edge events của Code Property Graph | Neo4j |
| `peft.source.metadata` | Chứa metadata cấp file: hash, số dòng, số node/edge, số hàm/class/import | Spark Structured Streaming -> MongoDB |
| `peft.parser.errors` | Ghi nhận lỗi parse để pipeline không dừng toàn bộ batch | Monitoring/debug |

Mỗi Kafka message dùng JSON payload và stable key. Với graph events, key là `element_id` hoặc `edge_id`; với metadata, key là `file_path`. Cách thiết kế này giúp replay cùng một source file không tạo bản ghi trùng ở sink nếu sink dùng upsert/MERGE theo key.

Các trường chính trong event gồm:

```text
schema_version
event_time
repo
commit_sha
file_path
element_id / edge_id / content_hash
properties
```


In [ ]:
print("Kafka topic commands:")
show_file(EVIDENCE_DIR / "kafka_topic_commands.txt", max_chars=4000)

Kafka topic commands:
kafka-topics --create --topic peft.cpg.nodes --bootstrap-server localhost:9092 --partitions 6 --replication-factor 1
kafka-topics --create --topic peft.cpg.edges --bootstrap-server localhost:9092 --partitions 6 --replication-factor 1
kafka-topics --create --topic peft.source.metadata --bootstrap-server localhost:9092 --partitions 3 --replication-factor 1
kafka-topics --create --topic peft.parser.errors --bootstrap-server localhost:9092 --partitions 3 --replication-factor 1



In [ ]:
print("Kafka topics describe:")
show_file(EVIDENCE_DIR / "kafka_topics_describe.txt", max_chars=6000)

Kafka topics describe:
Topic: peft.cpg.nodes	TopicId: o6KUKi7oSe21UB66IHghkQ	PartitionCount: 6	ReplicationFactor: 1	Configs: 
	Topic: peft.cpg.nodes	Partition: 0	Leader: 1	Replicas: 1	Isr: 1
	Topic: peft.cpg.nodes	Partition: 1	Leader: 1	Replicas: 1	Isr: 1
	Topic: peft.cpg.nodes	Partition: 2	Leader: 1	Replicas: 1	Isr: 1
	Topic: peft.cpg.nodes	Partition: 3	Leader: 1	Replicas: 1	Isr: 1
	Topic: peft.cpg.nodes	Partition: 4	Leader: 1	Replicas: 1	Isr: 1
	Topic: peft.cpg.nodes	Partition: 5	Leader: 1	Replicas: 1	Isr: 1
Topic: peft.cpg.edges	TopicId: 4YYB47O0TYSYcaWctvliHw	PartitionCount: 6	ReplicationFactor: 1	Configs: 
	Topic: peft.cpg.edges	Partition: 0	Leader: 1	Replicas: 1	Isr: 1
	Topic: peft.cpg.edges	Partition: 1	Leader: 1	Replicas: 1	Isr: 1
	Topic: peft.cpg.edges	Partition: 2	Leader: 1	Replicas: 1	Isr: 1
	Topic: peft.cpg.edges	Partition: 3	Leader: 1	Replicas: 1	Isr: 1
	Topic: peft.cpg.edges	Partition: 4	Leader: 1	Replicas: 1	Isr: 1
	Topic: peft.cpg.edges	Partition: 5	Leader: 1	Replicas: 

In [ ]:
sample_events = read_json(EVIDENCE_DIR / "sample_events_by_topic.json")
pprint(sample_events)

{'peft.cpg.edges': {'key': '2bafb589ec0db60d6e90f9110e241eedd61df446',
                    'topic': 'peft.cpg.edges',
                    'value': {'commit_sha': '79f4c362248d3b3b4bc2ed24704ed3183528c53f',
                              'edge_id': '2bafb589ec0db60d6e90f9110e241eedd61df446',
                              'edge_type': 'AST_CHILD',
                              'event_time': '2026-07-13T03:58:52.833413Z',
                              'file_path': '/Users/hvpu/Downloads/group4/peft/docs/source/_config.py',
                              'properties': {},
                              'repo': 'huggingface/peft',
                              'schema_version': 1,
                              'source_id': 'f289069d7149c9642c7fc928087221017d8db69a',
                              'target_id': 'd68235b762b56a51aa30be942f1e8702d2327c48'}},
 'peft.cpg.nodes': {'key': 'f289069d7149c9642c7fc928087221017d8db69a',
                    'topic': 'peft.cpg.nodes',
                    'val

### Kết quả Task 3

Kafka đã được dùng như event bus thật cho pipeline. Bốn topic trên được tạo trong Docker Kafka, sau đó parser publish thử event mẫu vào đúng topic tương ứng. Kết quả này xác nhận ba điểm quan trọng trước khi nối sang các sink:

- Producer ghi được node, edge và metadata events vào Kafka.
- Event được route đúng theo loại dữ liệu, giúp Neo4j và MongoDB consume độc lập.
- Stable key đã có sẵn trong message để phục vụ replay và idempotent ingestion ở các bước sau.


## 4. Graph Topology Ingestion into Neo4j

Theo yêu cầu của lab, node và edge của CPG phải được ghi vào **Neo4j trực tiếp từ Kafka**, không đi qua Spark.

Nhóm đã bổ sung Kafka Connect vào `docker-compose.yml`, cài **Neo4j Kafka Connector thật** trong container, và tạo hai connector:

```text
configs/neo4j/node-sink-connector.json
configs/neo4j/edge-sink-connector.json
```

Connector sử dụng class:

```text
org.neo4j.connectors.kafka.sink.Neo4jConnector
```

Neo4j sử dụng `MERGE` và constraint để đảm bảo idempotent ingestion. Khi publish lại cùng event, node/edge không bị nhân đôi.

**Phạm vi live Neo4j demo:** full parser đã sinh graph cho 365 files, nhưng phần live Neo4j graph ingestion được chạy trên file replay đại diện `loraplus.py`. Lý do là full graph có hơn 604k graph events (`250,837` nodes + `353,327` edges), khá nặng cho Docker local. Để tránh nói quá trong báo cáo, nhóm ghi rõ: full graph được parse offline, còn Neo4j Connector Sink live được chứng minh bằng representative replay file.


In [8]:
print("Kafka Connect plugins:")
show_file(EVIDENCE_DIR / "replay_end_to_end" / "connect_plugins_final.json", max_chars=6000)

print("\nNode connector status:")
show_file(EVIDENCE_DIR / "replay_end_to_end" / "node_connector_status_final.json", max_chars=4000)

print("\nEdge connector status:")
show_file(EVIDENCE_DIR / "replay_end_to_end" / "edge_connector_status_final.json", max_chars=4000)


Kafka Connect plugins:
[{"class":"org.neo4j.connectors.kafka.sink.Neo4jConnector","type":"sink","version":"5.5.0"},{"class":"org.apache.kafka.connect.mirror.MirrorCheckpointConnector","type":"source","version":"7.6.1-ccs"},{"class":"org.apache.kafka.connect.mirror.MirrorHeartbeatConnector","type":"source","version":"7.6.1-ccs"},{"class":"org.apache.kafka.connect.mirror.MirrorSourceConnector","type":"source","version":"7.6.1-ccs"},{"class":"org.neo4j.connectors.kafka.source.Neo4jConnector","type":"source","version":"5.5.0"}]

Node connector status:
{"name":"group4-peft-node-sink","connector":{"state":"RUNNING","worker_id":"connect:8083"},"tasks":[{"id":0,"state":"RUNNING","worker_id":"connect:8083"}],"type":"sink"}

Edge connector status:
{"name":"group4-peft-edge-sink","connector":{"state":"RUNNING","worker_id":"connect:8083"},"tasks":[{"id":0,"state":"RUNNING","worker_id":"connect:8083"}],"type":"sink"}


In [9]:
print("Neo4j constraints:")
show_file(EVIDENCE_DIR / "neo4j_constraints.txt", max_chars=4000)


Neo4j constraints:
name, type
"cpg_edge_id", "RELATIONSHIP_UNIQUENESS"
"cpg_node_id", "UNIQUENESS"



In [10]:
e2e_dir = EVIDENCE_DIR / "replay_end_to_end"

print("Neo4j before replay qua Kafka Connector Sink:")
show_file(e2e_dir / "neo4j_before_graph_connector.txt", max_chars=2000)

print("\nNeo4j sau khi publish lại before lần 2:")
show_file(e2e_dir / "neo4j_after_republish_before_connector.txt", max_chars=2000)

print("\nNeo4j after replay qua Kafka Connector Sink:")
show_file(e2e_dir / "neo4j_after_replay_connector.txt", max_chars=2000)

print("\nSupplementary manual MERGE idempotence check:")
show_file(EVIDENCE_DIR / "neo4j_counts_after_first_ingest.txt", max_chars=1000)
show_file(EVIDENCE_DIR / "neo4j_counts_after_second_ingest.txt", max_chars=1000)


Neo4j before replay qua Kafka Connector Sink:
nodes, edges
177, 292


Neo4j sau khi publish lại before lần 2:
nodes, edges
177, 292


Neo4j after replay qua Kafka Connector Sink:
nodes, edges
199, 338


Supplementary manual MERGE idempotence check:
nodes, edges
177, 292

nodes, edges
177, 292



### Kết quả Task 4

Neo4j Kafka Connector Sink thật đã chạy với file replay đại diện `loraplus.py`.

Kết quả qua connector thật:

```text
Before replay:              177 nodes, 292 edges
Republish before lần 2:     177 nodes, 292 edges
After replay:               199 nodes, 338 edges
```

Ý nghĩa:

- Luồng ghi graph là **Kafka → Neo4j Kafka Connector Sink → Neo4j**.
- Publish lại cùng dữ liệu không làm tăng số node/edge → idempotent OK.
- Publish bản after làm graph thay đổi → downstream phản ánh thay đổi của file.

> Ghi chú phạm vi: nhóm **chưa ingest full graph của 365 files vào Neo4j live**. Full graph đã được parse offline để chứng minh parser scalability; Neo4j live ingestion dùng representative replay file để giữ Docker local ổn định. Metadata full 365 files được stream live sang MongoDB ở Task 5.


## 5. Source Metadata Ingestion into MongoDB bằng Spark Structured Streaming

Metadata của source file được xử lý bằng luồng riêng:

```text
Kafka topic peft.source.metadata
→ Spark Structured Streaming
→ MongoDB collection
```

Spark job sử dụng checkpoint location để có thể resume từ offset đã xử lý, tránh đọc lại dữ liệu cũ sau khi restart.

Nhóm đã chạy hai mức kiểm tra:

1. Metadata sample/replay.
2. Full metadata cho toàn bộ 365 file Python.


In [11]:
print("Mongo metadata sample:")
show_file(EVIDENCE_DIR / "mongo_metadata_sample.txt", max_chars=4000)

print("\nSpark Mongo metadata sample:")
show_file(EVIDENCE_DIR / "spark_mongo_metadata_sample.txt", max_chars=4000)


Mongo metadata sample:
[
  {
    file_path: 'peft/src/peft/optimizers/loraplus.py',
    ast_node_count: 233,
    ast_edge_count: 227,
    cfg_edge_count: 25,
    dfg_edge_count: 50,
    call_edge_count: 19
  }
]


Spark Mongo metadata sample:
[
  {
    file_path: 'peft/src/peft/optimizers/loraplus.py',
    ast_node_count: Long('233'),
    ast_edge_count: Long('227'),
    cfg_edge_count: Long('25'),
    dfg_edge_count: Long('50'),
    call_edge_count: Long('19')
  }
]



### Full metadata live cho 365 file

Nhóm đã extract toàn bộ metadata từ `output/peft-events.jsonl`:

```text
365 metadata events
365 unique file paths
```

Sau đó publish vào Kafka topic `peft.source.metadata` và chạy Spark Structured Streaming `availableNow` để ghi vào MongoDB.


In [12]:
full_metadata_dir = EVIDENCE_DIR / "full_metadata"

print("Full metadata extract summary:")
pprint(read_json(full_metadata_dir / "full_metadata_extract_summary.json"))

print("\nKafka offsets sau khi publish 365 metadata events:")
show_file(full_metadata_dir / "kafka_full_metadata_offsets.txt", max_chars=4000)

print("\nSpark full metadata log:")
show_file(full_metadata_dir / "spark_full_metadata.log", max_chars=6000)

print("\nMongo full metadata count + sample:")
show_file(full_metadata_dir / "mongo_full_metadata_count_sample.json", max_chars=6000)

print("\nSpark checkpoint files:")
show_file(full_metadata_dir / "spark_full_metadata_checkpoint_files.txt", max_chars=4000)


Full metadata extract summary:
{'metadata_events': 365,
 'nul_bytes': 0,
 'source': 'output/peft-events.jsonl',
 'target': 'evidence/full_metadata/peft.source.metadata.full.jsonl',
 'unique_file_paths': 365}

Kafka offsets sau khi publish 365 metadata events:
peft.source.metadata:0:67
peft.source.metadata:1:180
peft.source.metadata:2:118


Spark full metadata log:
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/13 12:01:50 WARN Utils: Your hostname, Uyens-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.81 instead (on interface en0)
26/07/13 12:01:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/hvpu/Library/Python/3.9/lib/python/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /tmp/group4-ivy/cache
The jars for the packages stored in: /tmp/group4-ivy/jars
org.apache.spark#spark-sql-kafka

### Kết quả Task 5

Kết quả full metadata live:

```text
Kafka offsets:
partition 0: 67
partition 1: 180
partition 2: 118
Tổng: 365

Spark:
numInputRows: 365
numOutputRows: 365

MongoDB:
collection lab4.peft_metadata_full có 365 documents
```

Như vậy, toàn bộ metadata của 365 file Python đã được stream từ Kafka sang MongoDB thông qua Spark Structured Streaming.


## 6. Idempotent Replay Verification

Để kiểm tra replay, nhóm chọn file:

```text
src/peft/optimizers/loraplus.py
```

Nhóm thực hiện các bước:

1. Lưu bản gốc của file.
2. Publish bản before vào Kafka.
3. Publish lại bản before lần 2 để kiểm tra duplicate.
4. Sửa file và publish bản after.
5. Kiểm tra Neo4j và MongoDB sau replay.
6. Khôi phục file PEFT về trạng thái gốc sau khi kiểm tra.

Kết quả parser-level replay:

```text
Before: 470 events
After: 512 events
Stable IDs: 444
Added: 68
Removed: 26
```

Điều này cho thấy khi file thay đổi, phần lớn ID vẫn ổn định, đồng thời các node/edge mới và bị xóa được phát hiện đúng.


In [13]:
replay_dir = EVIDENCE_DIR / "replay"

print("Replay evidence files:")
if replay_dir.exists():
    for p in sorted(replay_dir.iterdir()):
        print("-", p.relative_to(PROJECT_ROOT))
else:
    print("[MISSING]", replay_dir)


Replay evidence files:
- evidence/replay/loraplus_after.py
- evidence/replay/loraplus_after_events.jsonl
- evidence/replay/loraplus_after_parse_summary.json
- evidence/replay/loraplus_before.py
- evidence/replay/loraplus_replay_summary.json


### Replay end-to-end qua Neo4j Kafka Connector Sink

Nhóm chạy replay qua luồng thật:

```text
Kafka → Neo4j Kafka Connector Sink → Neo4j
```

Kết quả:

```text
Before replay:              177 nodes, 292 edges
Republish before lần 2:     177 nodes, 292 edges
After replay:               199 nodes, 338 edges
```

Ý nghĩa:

- Publish lại cùng bản before không làm tăng node/edge → idempotent OK.
- Publish bản after làm graph thay đổi từ 177/292 lên 199/338 → graph được cập nhật theo nội dung file mới.
- Đây là live representative graph replay, không phải full 365-file graph ingest.


In [14]:
e2e_dir = EVIDENCE_DIR / "replay_end_to_end"

print("Neo4j before replay:")
show_file(e2e_dir / "neo4j_before_graph_connector.txt", max_chars=2000)

print("\nNeo4j after republish before:")
show_file(e2e_dir / "neo4j_after_republish_before_connector.txt", max_chars=2000)

print("\nNeo4j after replay:")
show_file(e2e_dir / "neo4j_after_replay_connector.txt", max_chars=2000)


Neo4j before replay:
nodes, edges
177, 292


Neo4j after republish before:
nodes, edges
177, 292


Neo4j after replay:
nodes, edges
199, 338



### Replay metadata qua Spark Streaming và MongoDB

Nhóm cũng stream metadata replay qua:

```text
Kafka → Spark Structured Streaming → MongoDB
```

Metadata trước replay:

```text
ast_node_count: 233
ast_edge_count: 227
cfg_edge_count: 25
dfg_edge_count: 50
call_edge_count: 19
function_count: 1
```

Metadata sau replay:

```text
ast_node_count: 252
ast_edge_count: 247
cfg_edge_count: 26
dfg_edge_count: 50
call_edge_count: 21
function_count: 2
```

Sự thay đổi này cho thấy metadata trong MongoDB đã được cập nhật theo phiên bản mới của file.


In [15]:
print("Mongo metadata before replay:")
show_file(e2e_dir / "mongo_metadata_before_replay.txt", max_chars=4000)

print("\nMongo metadata after replay:")
show_file(e2e_dir / "mongo_metadata_after_replay.txt", max_chars=4000)

print("\nSpark replay before log:")
show_file(e2e_dir / "spark_replay_before.log", max_chars=4000)

print("\nSpark replay after log:")
show_file(e2e_dir / "spark_replay_after.log", max_chars=4000)

print("\nSpark replay checkpoint files:")
show_file(e2e_dir / "spark_replay_checkpoint_files.txt", max_chars=4000)


Mongo metadata before replay:
[
  {
    file_path: 'peft/src/peft/optimizers/loraplus.py',
    content_hash: 'cd3668016860937f1b8af7246cf069824f6dc6b6',
    ast_node_count: Long('233'),
    ast_edge_count: Long('227'),
    cfg_edge_count: Long('25'),
    dfg_edge_count: Long('50'),
    call_edge_count: Long('19'),
    function_count: Long('1')
  }
]


Mongo metadata after replay:
[
  {
    file_path: 'peft/src/peft/optimizers/loraplus.py',
    content_hash: 'cd3668016860937f1b8af7246cf069824f6dc6b6',
    ast_node_count: Long('233'),
    ast_edge_count: Long('227'),
    cfg_edge_count: Long('25'),
    dfg_edge_count: Long('50'),
    call_edge_count: Long('19'),
    function_count: Long('1')
  },
  {
    file_path: 'peft/src/peft/optimizers/loraplus.py',
    content_hash: '52c747571aa630118249b1a5429820fb48e3ebbe',
    ast_node_count: Long('252'),
    ast_edge_count: Long('247'),
    cfg_edge_count: Long('26'),
    dfg_edge_count: Long('50'),
    call_edge_count: Long('21'),
    function

### Kết quả Task 6

Replay verification đạt các yêu cầu chính:

| Kiểm tra | Kết quả |
|---|---|
| Reprocess cùng bản before | Không tạo duplicate trong Neo4j |
| Publish bản after | Neo4j graph thay đổi đúng |
| Metadata after | MongoDB cập nhật số node/edge/function |
| Spark checkpoint | Có commit `0` và `1`, chứng minh Spark resume bằng checkpoint |

Như vậy pipeline thỏa tính incremental và idempotent trong kịch bản replay.


## 7. Kiểm thử và môi trường chạy

Nhóm sử dụng Docker để chạy các service:

| Service | Port |
|---|---|
| Kafka | `localhost:9092` |
| Zookeeper | `localhost:2181` |
| Neo4j Browser | `localhost:7474` |
| Neo4j Bolt | `localhost:7687` |
| MongoDB | `localhost:27017` |

Ngoài ra, nhóm cài `pytest` và chạy unit/manual tests để kiểm tra các phần chính của code.


In [16]:
print("pytest result:")
show_file(EVIDENCE_DIR / "pytest.txt", max_chars=3000)

print("\nManual tests:")
show_file(EVIDENCE_DIR / "manual_tests.txt", max_chars=3000)

print("\nDocker compose final status:")
show_file(EVIDENCE_DIR / "full_metadata" / "docker_compose_ps_after_full_metadata.txt", max_chars=5000)


pytest result:
....                                                                     [100%]
4 passed in 7.08s


Manual tests:
manual parser tests passed
project_python_files=14
sample_nodes=58
sample_edges=78
sample_replay_stable=135


Docker compose final status:
NAME                 IMAGE                                 COMMAND                  SERVICE     CREATED          STATUS                      PORTS
group4-connect-1     confluentinc/cp-kafka-connect:7.6.1   "bash -lc 'confluent…"   connect     22 minutes ago   Up 22 minutes (unhealthy)   0.0.0.0:8083->8083/tcp, [::]:8083->8083/tcp
group4-kafka-1       confluentinc/cp-kafka:7.6.1           "/etc/confluent/dock…"   kafka       45 minutes ago   Up 45 minutes               0.0.0.0:9092->9092/tcp, [::]:9092->9092/tcp
group4-mongo-1       mongo:7.0                             "docker-entrypoint.s…"   mongo       45 minutes ago   Up 45 minutes               0.0.0.0:27017->27017/tcp, [::]:27017->27017/tcp
group4-neo4j-1       neo4j

## 8. Cấu trúc source code và config

Các thành phần chính của project:

```text
group4/
├── peft/                         # repository PEFT được clone
├── configs/
│   ├── generated/                # generated configs
│   └── neo4j/                    # Neo4j Kafka Sink connector configs
├── evidence/                     # evidence logs, JSON, query outputs
├── jobs/                         # Spark Structured Streaming jobs
├── output/                       # full event JSONL
├── docker-compose.yml            # Kafka, Neo4j, MongoDB, Kafka Connect
└── docs/book/                    # Jupyter Book report
```

Các file evidence quan trọng:

```text
evidence/discover_no_tests.json
evidence/parse_summary.json
evidence/sample_events_by_topic.json
evidence/kafka_topics_describe.txt
evidence/replay_end_to_end/node_connector_status_final.json
evidence/replay_end_to_end/edge_connector_status_final.json
evidence/replay_end_to_end/neo4j_after_replay_connector.txt
evidence/full_metadata/spark_full_metadata.log
evidence/full_metadata/mongo_full_metadata_count_sample.json
```


## 9. Hướng dẫn chạy lại pipeline

Các lệnh dưới đây phản ánh đúng project hiện tại.

```bash
# 1. Khởi động hạ tầng: Kafka, Zookeeper, Neo4j, MongoDB, Kafka Connect
docker compose up -d

# 2. Cài Python deps nếu cần
python3 -m pip install --user pytest kafka-python pyspark

# 3. Clone PEFT nếu chưa có
git clone --depth 1 https://github.com/huggingface/peft.git peft

# 4. Discovery repository
PYTHONPATH=src python3 -m group4_lab.cli discover --repo peft

# 5. Parse toàn bộ PEFT offline ra JSONL
PYTHONPATH=src python3 -m group4_lab.cli parse-repo \
  --repo peft \
  --repo-name huggingface/peft \
  --commit-sha 79f4c362248d3b3b4bc2ed24704ed3183528c53f \
  --publisher console \
  --publisher-output output/peft-events.jsonl

# 6. Tạo Kafka topics
python3 scripts/create_topics.py
# Sau đó chạy các lệnh kafka-topics được in ra trong container kafka.

# 7. Tạo Neo4j Kafka Connector Sink
curl -X POST http://localhost:8083/connectors \
  -H "Content-Type: application/json" \
  --data @configs/neo4j/node-sink-connector.json

curl -X POST http://localhost:8083/connectors \
  -H "Content-Type: application/json" \
  --data @configs/neo4j/edge-sink-connector.json

# 8. Chạy Spark Structured Streaming metadata sang MongoDB
JAVA_HOME=/tmp/group4-jdk17/Contents/Home \
GROUP4_SPARK_CHECKPOINT=checkpoints/full_metadata_stream \
GROUP4_MONGO_COLLECTION=peft_metadata_full \
spark-submit \
  --conf spark.jars.ivy=/tmp/group4-ivy \
  --packages org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.3,org.mongodb.spark:mongo-spark-connector_2.13:10.5.0 \
  jobs/mongo_streaming_available_now.py

# 9. Kiểm tra tests
PYTHONPATH=src python3 -m pytest -q
```

Trong lần chạy thực tế của nhóm, các output và query result được lưu vào `evidence/` để đưa vào báo cáo.


## 10. Kết luận

Nhóm đã xây dựng được pipeline xử lý mã nguồn Python theo hướng streaming cho repository `huggingface/peft`.

Các kết quả chính:

- Clone và discovery thành công 365 file Python không tính tests.
- Parse toàn bộ PEFT thành:
  - 250,837 node events.
  - 353,327 edge events.
  - 365 metadata events.
  - 0 parser errors.
- Thiết kế và tạo Kafka topics cho nodes, edges, metadata và errors.
- Cài đặt Neo4j Kafka Connector Sink thật và ghi graph vào Neo4j theo hướng idempotent trên replay file đại diện `loraplus.py`.
- Stream full metadata của 365 file qua Kafka và Spark Structured Streaming vào MongoDB, kết quả MongoDB có 365 documents.
- Thực hiện replay end-to-end với `loraplus.py`, chứng minh:
  - Reprocess cùng dữ liệu không tạo duplicate trong Neo4j.
  - Khi file thay đổi, Neo4j graph thay đổi đúng.
  - MongoDB metadata before/after phản ánh số node/edge/function thay đổi.
  - Spark checkpoint hoạt động qua các lần chạy.

Giới hạn được ghi rõ: nhóm chưa ingest live toàn bộ full graph 365 files vào Neo4j vì workload có hơn 604k graph events. Thay vào đó, full graph được parse offline để chứng minh parser scalability; Neo4j live Connector Sink được chứng minh bằng representative replay file, còn full metadata 365 files được stream live sang MongoDB.

Nhìn chung, pipeline đáp ứng các yêu cầu chính của Lab 04: incremental parsing, Kafka topic design, Neo4j graph ingestion bằng Kafka Connector Sink, MongoDB metadata ingestion bằng Spark Structured Streaming, và idempotent replay verification.


## 11. Reflection

Trong quá trình thực hiện, nhóm gặp một số vấn đề kỹ thuật:

1. **Docker daemon chưa chạy**  
   Ban đầu không thể khởi động Kafka/Neo4j/MongoDB vì Docker Desktop chưa chạy. Sau khi mở Docker Desktop và chạy lại `docker compose up -d`, các service hoạt động bình thường.

2. **Python 3.9 không hỗ trợ đầy đủ `dataclass(slots=True)`**  
   Nhóm bỏ `slots=True` để code tương thích với Python 3.9 trong môi trường hiện tại.

3. **Replay bị đổi ID hàng loạt khi dùng file backup**  
   Nhóm sửa replay để so sánh cùng logical file path, nhờ đó stable IDs phản ánh đúng thay đổi nội dung file thay vì thay đổi đường dẫn.

4. **Java 25 không tương thích khi chạy Spark**  
   Nhóm sử dụng JDK 17 portable trong `/tmp` để chạy PySpark ổn định.

5. **Kafka topic bị dính record lỗi từ lần producer timeout**  
   Nhóm clean/recreate các topic `peft.cpg.*` để đảm bảo evidence cuối phản ánh pipeline sạch.

Các vấn đề trên đều đã được ghi lại trong evidence và được xử lý trước khi viết báo cáo.
